# 05 · PyTorch MLP

Goal:

1. Learn the basic deep-learning training loop using PyTorch.
2. Use the same chronological evaluation design as Logistic Regression and XGBoost.
3. Use 2024 only for internal early stopping.
4. Retrain on all 2018 to 2024 training data.
5. Evaluate once on 2025.
6. Keep 2026 untouched.

Model: `24 inputs → 64 → 32 → 1 logit`.

In [ ]:
from google.colab import drive
from pathlib import Path

import copy
import json
import random
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/ai-tech-market-risk')
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports' / 'metrics'
MODEL_DIR = PROJECT_ROOT / 'models'

for directory in [REPORTS_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ML_DATA_FILE = PROCESSED_DATA_DIR / 'ml_features.csv'
PREVIOUS_METRICS_FILE = REPORTS_DIR / '04_xgboost_validation_metrics.csv'
MLP_METRICS_FILE = REPORTS_DIR / '05_mlp_validation_metrics.csv'
MLP_MODEL_FILE = MODEL_DIR / 'mlp_candidate.pt'
MLP_PREPROCESSOR_FILE = MODEL_DIR / 'mlp_preprocessor.joblib'
MLP_METADATA_FILE = MODEL_DIR / 'mlp_candidate_metadata.json'

print('PyTorch:', torch.__version__)
print('scikit-learn:', sklearn.__version__)

## 1. Reproducibility and device

Colab will use a GPU if available. This model is small enough to run on CPU too.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Load and validate the processed dataset

In [ ]:
if not ML_DATA_FILE.exists():
    raise FileNotFoundError(f'Processed dataset not found: {ML_DATA_FILE}. Run Notebook 02 first.')

ml_data = pd.read_csv(ML_DATA_FILE, parse_dates=['Date'])
ml_data = ml_data.sort_values(['Date', 'Ticker']).reset_index(drop=True)

FEATURE_COLUMNS = [
    'Return_1D','Return_5D','Return_10D','Return_20D',
    'Volatility_5D','Volatility_10D','Volatility_20D',
    'Volume_Change_1D','Relative_Volume_20D',
    'Price_vs_MA_5D','Price_vs_MA_20D',
    'SPY_Return_1D','QQQ_Return_1D','SMH_Return_1D',
    'SPY_Return_5D','QQQ_Return_5D','SMH_Return_5D',
    'Excess_vs_QQQ_1D','Excess_vs_SMH_1D',
]
CATEGORICAL_COLUMNS = ['Ticker']
MODEL_COLUMNS = FEATURE_COLUMNS + CATEGORICAL_COLUMNS
TARGET_COLUMN = 'Large_Move_5D'
TARGET_HORIZON_DAYS = 5

required_columns = ['Date', TARGET_COLUMN] + MODEL_COLUMNS
missing_columns = set(required_columns) - set(ml_data.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
if ml_data[MODEL_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError('Missing model values detected.')
if np.isinf(ml_data[FEATURE_COLUMNS].to_numpy()).any():
    raise ValueError('Infinite numeric features detected.')

print('Shape:', ml_data.shape)
print('Date range:', ml_data['Date'].min().date(), 'to', ml_data['Date'].max().date())

## 3. Recreate the same chronological split

In [ ]:
VALIDATION_START = pd.Timestamp('2025-01-01')
TEST_START = pd.Timestamp('2026-01-01')

def purge_last_trading_dates(data, n_dates):
    unique_dates = np.array(sorted(data['Date'].unique()))
    if len(unique_dates) <= n_dates:
        raise ValueError('Not enough dates to apply purge.')
    purged_dates = unique_dates[-n_dates:]
    cleaned = data[~data['Date'].isin(purged_dates)].copy()
    return cleaned, purged_dates

train_data = ml_data[ml_data['Date'] < VALIDATION_START].copy()
validation_data = ml_data[(ml_data['Date'] >= VALIDATION_START) & (ml_data['Date'] < TEST_START)].copy()
test_data = ml_data[ml_data['Date'] >= TEST_START].copy()

train_data, purged_train_dates = purge_last_trading_dates(train_data, TARGET_HORIZON_DAYS)
validation_data, purged_validation_dates = purge_last_trading_dates(validation_data, TARGET_HORIZON_DAYS)

split_summary = pd.DataFrame({
    'split':['train','validation','test'],
    'rows':[len(train_data),len(validation_data),len(test_data)],
    'start_date':[train_data['Date'].min(), validation_data['Date'].min(), test_data['Date'].min()],
    'end_date':[train_data['Date'].max(), validation_data['Date'].max(), test_data['Date'].max()],
    'positive_rate':[
        train_data[TARGET_COLUMN].mean()*100,
        validation_data[TARGET_COLUMN].mean()*100,
        np.nan,
    ],
})

if not (train_data['Date'].max() < validation_data['Date'].min() < test_data['Date'].min()):
    raise ValueError('Chronological split ordering failed.')

print('2026 target prevalence intentionally hidden until final evaluation.')
split_summary

## 4. Internal early-stopping split

2018 to 2023 is used for temporary training. 2024 is used only to decide when to stop training. 2025 remains the external validation year.

In [ ]:
EARLY_STOP_START = pd.Timestamp('2024-01-01')

temporary_fit_data = train_data[train_data['Date'] < EARLY_STOP_START].copy()
early_stop_data = train_data[train_data['Date'] >= EARLY_STOP_START].copy()
temporary_fit_data, internal_purged_dates = purge_last_trading_dates(temporary_fit_data, TARGET_HORIZON_DAYS)

internal_summary = pd.DataFrame({
    'split':['temporary_fit','early_stop'],
    'rows':[len(temporary_fit_data),len(early_stop_data)],
    'start_date':[temporary_fit_data['Date'].min(), early_stop_data['Date'].min()],
    'end_date':[temporary_fit_data['Date'].max(), early_stop_data['Date'].max()],
    'positive_rate':[
        temporary_fit_data[TARGET_COLUMN].mean()*100,
        early_stop_data[TARGET_COLUMN].mean()*100,
    ],
})

print('Internal purged dates:', pd.to_datetime(internal_purged_dates).date)
internal_summary

## 5. Preprocessing

Neural networks benefit from scaled numeric inputs, so we standardize all numeric features. Ticker is one-hot encoded.

In [ ]:
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('numeric', StandardScaler(), FEATURE_COLUMNS),
            ('ticker', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_COLUMNS),
        ],
        sparse_threshold=0,
    )

early_preprocessor = build_preprocessor()
X_temp_fit = early_preprocessor.fit_transform(temporary_fit_data[MODEL_COLUMNS]).astype(np.float32)
X_early_stop = early_preprocessor.transform(early_stop_data[MODEL_COLUMNS]).astype(np.float32)
y_temp_fit = temporary_fit_data[TARGET_COLUMN].to_numpy(dtype=np.float32).reshape(-1,1)
y_early_stop = early_stop_data[TARGET_COLUMN].to_numpy(dtype=np.float32).reshape(-1,1)

print('Input features:', X_temp_fit.shape[1])
print('Temporary fit:', X_temp_fit.shape)
print('Early stop:', X_early_stop.shape)

## 6. Convert data to tensors and batches

In [ ]:
BATCH_SIZE = 128
train_dataset = TensorDataset(torch.from_numpy(X_temp_fit), torch.from_numpy(y_temp_fit))
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
X_early_stop_tensor = torch.from_numpy(X_early_stop).to(device)
y_early_stop_tensor = torch.from_numpy(y_early_stop).to(device)
print('Training batches:', len(train_loader))

## 7. Define the MLP

`Linear` layers learn weighted combinations of inputs. `ReLU` adds non-linearity. `Dropout` reduces overfitting. The final layer returns one logit.

In [ ]:
class MarketRiskMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)

## 8. Train with early stopping

In [ ]:
MAX_EPOCHS = 200
PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

torch.manual_seed(SEED)
model = MarketRiskMLP(X_temp_fit.shape[1]).to(device)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history = {'train_loss': [], 'early_stop_loss': []}
best_state = None
best_loss = float('inf')
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    total_train_rows = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = loss_function(logits, batch_y)
        loss.backward()
        optimizer.step()

        batch_size = batch_X.size(0)
        total_train_loss += loss.item() * batch_size
        total_train_rows += batch_size

    train_loss = total_train_loss / total_train_rows

    model.eval()
    with torch.no_grad():
        early_stop_logits = model(X_early_stop_tensor)
        early_stop_loss = loss_function(early_stop_logits, y_early_stop_tensor).item()

    history['train_loss'].append(train_loss)
    history['early_stop_loss'].append(early_stop_loss)

    if early_stop_loss < best_loss - 1e-5:
        best_loss = early_stop_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0 or epochs_without_improvement >= PATIENCE:
        print(f'Epoch {epoch:03d} | train={train_loss:.4f} | early_stop={early_stop_loss:.4f}')

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

if best_state is None:
    raise RuntimeError('Training never produced a best model.')

print('Best epoch:', best_epoch)
print('Best early-stop loss:', round(best_loss, 6))

## 9. Plot training loss

In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)
plt.figure(figsize=(10,5))
plt.plot(epochs, history['train_loss'], label='Training loss')
plt.plot(epochs, history['early_stop_loss'], label='2024 early-stop loss')
plt.axvline(best_epoch, linestyle=':', label=f'Best epoch = {best_epoch}')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy loss')
plt.title('PyTorch MLP Training')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Retrain from scratch on all 2018 to 2024 training data

We now know how long to train. A fresh preprocessor and fresh network are trained using the full training period for exactly `best_epoch` epochs.

In [ ]:
final_preprocessor = build_preprocessor()
X_train = final_preprocessor.fit_transform(train_data[MODEL_COLUMNS]).astype(np.float32)
X_validation = final_preprocessor.transform(validation_data[MODEL_COLUMNS]).astype(np.float32)
X_test = final_preprocessor.transform(test_data[MODEL_COLUMNS]).astype(np.float32)

y_train = train_data[TARGET_COLUMN].to_numpy(dtype=np.float32).reshape(-1,1)
y_validation = validation_data[TARGET_COLUMN].to_numpy(dtype=np.int64)
y_test = test_data[TARGET_COLUMN].to_numpy(dtype=np.int64)

final_train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)

print('Full train:', X_train.shape)
print('Validation:', X_validation.shape)
print('Test prepared:', X_test.shape)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

final_model = MarketRiskMLP(X_train.shape[1]).to(device)
final_loss_function = nn.BCEWithLogitsLoss()
final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

for epoch in range(1, best_epoch + 1):
    final_model.train()
    total_loss = 0.0
    total_rows = 0

    for batch_X, batch_y in final_train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        final_optimizer.zero_grad()
        logits = final_model(batch_X)
        loss = final_loss_function(logits, batch_y)
        loss.backward()
        final_optimizer.step()

        batch_size = batch_X.size(0)
        total_loss += loss.item() * batch_size
        total_rows += batch_size

    if epoch == 1 or epoch % 10 == 0 or epoch == best_epoch:
        print(f'Final training epoch {epoch:03d}/{best_epoch} | loss={total_loss/total_rows:.4f}')

## 11. Evaluate once on 2025 validation

In [ ]:
final_model.eval()
X_validation_tensor = torch.from_numpy(X_validation).to(device)

with torch.no_grad():
    validation_logits = final_model(X_validation_tensor).squeeze(1)
    validation_probabilities = torch.sigmoid(validation_logits).cpu().numpy()

validation_predictions = (validation_probabilities >= 0.5).astype(int)

mlp_metrics = {
    'model':'PyTorchMLP',
    'accuracy':accuracy_score(y_validation, validation_predictions),
    'precision':precision_score(y_validation, validation_predictions, zero_division=0),
    'recall':recall_score(y_validation, validation_predictions, zero_division=0),
    'f1':f1_score(y_validation, validation_predictions, zero_division=0),
    'roc_auc':roc_auc_score(y_validation, validation_probabilities),
    'pr_auc':average_precision_score(y_validation, validation_probabilities),
}

mlp_results = pd.DataFrame([mlp_metrics]).set_index('model')
mlp_results.round(4)

## 12. Compare all models

In [ ]:
if not PREVIOUS_METRICS_FILE.exists():
    raise FileNotFoundError('Notebook 04 metrics not found. Run Notebook 04 first.')

previous_results = pd.read_csv(PREVIOUS_METRICS_FILE, index_col=0)
model_comparison = pd.concat([previous_results, mlp_results], axis=0)

display(model_comparison.round(4))

best_auc_model = model_comparison['roc_auc'].idxmax()
best_auc = model_comparison['roc_auc'].max()
print('Current best ROC AUC:', best_auc_model, round(best_auc, 4))

## 13. Classification report and confusion matrix

In [ ]:
print(classification_report(
    y_validation,
    validation_predictions,
    target_names=['Normal Move','Large Move'],
    digits=4,
))

cm = confusion_matrix(y_validation, validation_predictions)
print('Confusion matrix:')
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Normal Move','Large Move'],
)
disp.plot()
plt.title('PyTorch MLP - 2025 Validation')
plt.show()

## 14. Save model, preprocessor and metrics

The preprocessing object must be saved with the model because the neural network expects transformed inputs, not the raw feature table.

In [ ]:
model_comparison.to_csv(MLP_METRICS_FILE)

torch.save({
    'model_state_dict': final_model.state_dict(),
    'input_size': int(X_train.shape[1]),
}, MLP_MODEL_FILE)

joblib.dump(final_preprocessor, MLP_PREPROCESSOR_FILE)

metadata = {
    'pytorch_version': torch.__version__,
    'best_epoch': int(best_epoch),
    'best_internal_loss': float(best_loss),
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'seed': SEED,
    'architecture': [int(X_train.shape[1]), 64, 32, 1],
    'training_start': str(train_data['Date'].min().date()),
    'training_end': str(train_data['Date'].max().date()),
    'validation_start': str(validation_data['Date'].min().date()),
    'validation_end': str(validation_data['Date'].max().date()),
    'test_evaluated': False,
}

with open(MLP_METADATA_FILE, 'w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2)

for file_path in [MLP_METRICS_FILE, MLP_MODEL_FILE, MLP_PREPROCESSOR_FILE, MLP_METADATA_FILE]:
    print(file_path.name, 'exists:', file_path.exists(), 'size:', file_path.stat().st_size if file_path.exists() else None)

## 15. Reload verification

In [ ]:
reloaded_preprocessor = joblib.load(MLP_PREPROCESSOR_FILE)
reloaded_checkpoint = torch.load(MLP_MODEL_FILE, map_location=device)

reloaded_model = MarketRiskMLP(reloaded_checkpoint['input_size']).to(device)
reloaded_model.load_state_dict(reloaded_checkpoint['model_state_dict'])
reloaded_model.eval()

reloaded_X_validation = reloaded_preprocessor.transform(validation_data[MODEL_COLUMNS]).astype(np.float32)

with torch.no_grad():
    reloaded_probabilities = torch.sigmoid(
        reloaded_model(torch.from_numpy(reloaded_X_validation).to(device)).squeeze(1)
    ).cpu().numpy()

if not np.allclose(validation_probabilities, reloaded_probabilities, rtol=1e-5, atol=1e-6):
    raise ValueError('Reloaded MLP predictions differ.')

print('MLP reload verification passed.')
print('PyTorch MLP pipeline completed successfully.')
print('2026 test set remains untouched.')

# What to send back

Send:

1. Device used
2. `internal_summary`
3. Best epoch and best early-stop loss
4. `model_comparison`
5. PyTorch classification report
6. Confusion matrix

Then we will decide whether deep learning added value. After that, the fastest route is MLOps with MLflow. LSTM can remain an optional later experiment so it does not block project completion.